<a href="https://colab.research.google.com/github/jameshphan-png/Group-Exercise-Agentic-AI-in-Supply-Chain-/blob/dev/Group_Exercise_Agentic_AI_in_Supply_Chain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic AI on Supply Chain

The purpose of this Agent is to represent itself as an inventory replenishment agent that decides whether to **place an order or wait** based on expected demand, current stock, lead time, service level, and cost trade-offs.

IMPORTANT: MAKE SURE TO DOWNLOAD THE FOLLOWING FILES
 - inventory.xlsx
 - params.xlsx
 - sales.xlsx

## Agent role
The agent optimizes for:
- **Low stockouts** to protect customer service
- **Low total cost** by balancing holding cost against stockout cost

## Decision inputs
The agent uses a variety of factors when making orders or holding them:
- Forecasted daily demand
- On-hand inventory
- Purchase orders already in transit
- Lead time
- Service level
- Minimum order quantity
- Holding and stockout cost assumptions

## Decision output
At each review day, the agent returns one of two actions that determines the action to do:
- **ORDER** a quantity that respects the policy guardrails
- **WAIT** when inventory is already sufficient

## Trust, ethics, and guardrails
The design provides transparency by using a following ruleset, providing more realistic output and details to simulate an agent's workflow, based on:
- Safety stock is tied to the requested service level
- Lead time is explicitly considered
- Minimum order quantity is respected
- Over-ordering is capped to avoid unrealistic or wasteful decisions
- Daily logs explain *why* the agent ordered or waited

The purpose of having these rulesets is tied towards trust & transparency, as inventory decisions can influence for both:
- **Customer experience** (avoid stockouts)
- **Cost responsibility** (avoid unnecessary holding cost)


In [ ]:
# ================================================================
# 1. Imports and configuration
# ================================================================
# What this section does:
# - Imports the libraries used in the notebook
# - Sets the adjustable policy parameters
# - Defines simple switches for logging and file output
#
# Description:
# Keeping the configuration together makes the notebook easier to
# audit, tune, and explain. This is useful for both business trust
# and experimentation.

# Standard library imports support numeric calculations and file path checks.
import math
from pathlib import Path

# Third-party libraries handle arrays, tables, and service-level math.
import numpy as np
import pandas as pd
from scipy.stats import norm

# -------------------------
# User-adjustable settings
# -------------------------
FORECAST_METHOD = "ewma"      # "ewma" or "naive"
ALPHA = 0.35                  # EWMA smoothing weight
REVIEW_PERIOD_DAYS = 1        # how often the agent reviews inventory
ORDER_COST_PER_PO = 0.0       # can stay 0.0 for the assignment
OVERORDER_CAP_MULTIPLIER = 2.5

# Logging controls keep the notebook flexible: full logs for explanation,
# or lighter output when only a summary is needed.
# Logging controls
PRINT_FULL_DAILY_LOG = True
PRINT_SUMMARY_ONLY = False

# Output controls determine whether CSV artifacts are written at the end.
# Output controls
SAVE_OUTPUTS = True

# The notebook will try CSV first and then Excel if CSV is not present.
# Data file candidates
SALES_CANDIDATES = ["sales.csv", "sales.xlsx"]
INVENTORY_CANDIDATES = ["inventory.csv", "inventory.xlsx"]
PARAMS_CANDIDATES = ["params.csv", "params.xlsx"]

print("Configuration loaded.")
print(f"FORECAST_METHOD = {FORECAST_METHOD}")
print(f"ALPHA = {ALPHA}")
print(f"REVIEW_PERIOD_DAYS = {REVIEW_PERIOD_DAYS}")
print(f"ORDER_COST_PER_PO = {ORDER_COST_PER_PO}")
print(f"OVERORDER_CAP_MULTIPLIER = {OVERORDER_CAP_MULTIPLIER}")

Configuration loaded.
FORECAST_METHOD = ewma
ALPHA = 0.35
REVIEW_PERIOD_DAYS = 1
ORDER_COST_PER_PO = 0.0
OVERORDER_CAP_MULTIPLIER = 2.5


---

## 2. Data loading and demand aggregation

This section:
- Loads the three required files
- Supports both CSV and Excel
- Validates the required columns
- Aggregates daily demand per SKU

Required data layout:
- `sales.csv` / `sales.xlsx`: `date, sku, qty_sold`
- `inventory.csv` / `inventory.xlsx`: `sku, opening_stock`
- `params.csv` / `params.xlsx`: `sku, unit_cost, holding_cost_per_day, stockout_cost, lead_time_days, min_order_qty, service_level`


In [ ]:
# ================================================================
# 2. Data loading utilities
# ================================================================
# What this section does:
# - Finds the first available data file from CSV/Excel candidates
# - Loads each file into pandas
# - Validates that the schema matches the assignment
# - Aggregates demand by day and SKU
#
# Trust / governance angle:
# Explicit schema checks reduce silent errors. This is important because
# poor data quality can lead to poor replenishment decisions.

def find_existing_file(candidates):
    """Return the first existing file path from a list of candidate names."""
    for name in candidates:
        path = Path(name)
        if path.exists():
            return path
    raise FileNotFoundError(f"None of these files were found: {candidates}")

def load_table(path):
    """Load a CSV or Excel file based on the file extension."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file type: {path}")

def validate_columns(df, required_columns, table_name):
    """Check whether a dataframe contains all required columns."""
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")

def load_and_prepare_data():
    """Load sales, inventory, and parameter files, then clean and aggregate them."""
    # Locate whichever version of each file exists in the working directory.
    sales_path = find_existing_file(SALES_CANDIDATES)
    inventory_path = find_existing_file(INVENTORY_CANDIDATES)
    params_path = find_existing_file(PARAMS_CANDIDATES)

    # Load raw tables before validation and cleaning.
    sales_df = load_table(sales_path)
    inventory_df = load_table(inventory_path)
    params_df = load_table(params_path)

    validate_columns(sales_df, ["date", "sku", "qty_sold"], "sales")
    validate_columns(inventory_df, ["sku", "opening_stock"], "inventory")
    validate_columns(
        params_df,
        [
            "sku",
            "unit_cost",
            "holding_cost_per_day",
            "stockout_cost",
            "lead_time_days",
            "min_order_qty",
            "service_level",
        ],
        "params",
    )

    # Work on copies so the original loaded dataframes stay unchanged.
    sales_df = sales_df.copy()
    inventory_df = inventory_df.copy()
    params_df = params_df.copy()

    # Normalize data types so downstream calculations behave predictably.
    sales_df["date"] = pd.to_datetime(sales_df["date"])
    sales_df["sku"] = sales_df["sku"].astype(str)
    inventory_df["sku"] = inventory_df["sku"].astype(str)
    params_df["sku"] = params_df["sku"].astype(str)

    sales_df["qty_sold"] = pd.to_numeric(sales_df["qty_sold"], errors="coerce").fillna(0.0)
    inventory_df["opening_stock"] = pd.to_numeric(inventory_df["opening_stock"], errors="coerce")
    params_df["unit_cost"] = pd.to_numeric(params_df["unit_cost"], errors="coerce")
    params_df["holding_cost_per_day"] = pd.to_numeric(params_df["holding_cost_per_day"], errors="coerce")
    params_df["stockout_cost"] = pd.to_numeric(params_df["stockout_cost"], errors="coerce")
    params_df["lead_time_days"] = pd.to_numeric(params_df["lead_time_days"], errors="coerce")
    params_df["min_order_qty"] = pd.to_numeric(params_df["min_order_qty"], errors="coerce")
    params_df["service_level"] = pd.to_numeric(params_df["service_level"], errors="coerce")

    # Aggregate raw sales into the required daily demand signal per SKU.
    sales_daily = (
        sales_df.groupby(["date", "sku"], as_index=False)["qty_sold"]
        .sum()
        .sort_values(["date", "sku"])
        .reset_index(drop=True)
    )

    print("Loaded files successfully:")
    print(f"  sales file     : {sales_path}")
    print(f"  inventory file : {inventory_path}")
    print(f"  params file    : {params_path}")
    print()
    print("Aggregated daily demand by SKU.")
    print(f"Rows in sales_daily: {len(sales_daily)}")
    print(f"SKUs found: {sorted(sales_daily['sku'].unique().tolist())}")
    print(f"Date range: {sales_daily['date'].min().date()} to {sales_daily['date'].max().date()}")

    return sales_daily, inventory_df, params_df

# Execute the data preparation step once so later sections can reuse the cleaned inputs.
sales_daily, inventory_df, params_df = load_and_prepare_data()

print("\nSample of aggregated daily demand:")
print(sales_daily.head(10).to_string(index=False))

Loaded files successfully:
  sales file     : sales.xlsx
  inventory file : inventory.xlsx
  params file    : params.xlsx

Aggregated daily demand by SKU.
Rows in sales_daily: 270
SKUs found: ['SKU_A', 'SKU_B', 'SKU_C']
Date range: 2025-01-01 to 2025-03-31

Sample of aggregated daily demand:
      date   sku  qty_sold
2025-01-01 SKU_A        13
2025-01-01 SKU_B        10
2025-01-01 SKU_C         9
2025-01-02 SKU_A        11
2025-01-02 SKU_B         6
2025-01-02 SKU_C         4
2025-01-03 SKU_A        13
2025-01-03 SKU_B         9
2025-01-03 SKU_C         7
2025-01-04 SKU_A         9


---

## 3. Forecasting logic

The agent uses the following tools to simulate a mock forecast demand:
- **EWMA forecast**: smooths history while still adapting to change
- **Naive forecast**: tomorrow equals yesterday

The forecast is updated **every day** after observing the current day's demand.


In [ ]:
# ================================================================
# 3. Forecasting helpers
# ================================================================
# What this section does:
# - Creates the initial forecast for each SKU
# - Updates the forecast each day
# - Stores error history for safety stock estimation
#
# Description:
# Forecasting is the agent's view of the near future. It does not need
# to be perfect, but it must be consistent and explainable.

def build_daily_demand_grid(sales_daily, sku_list):
    """Create a full date-SKU grid so missing dates become zero demand days."""
    # Build a continuous calendar so zero-demand days are not accidentally skipped.
    all_dates = pd.date_range(sales_daily["date"].min(), sales_daily["date"].max(), freq="D")
    full_index = pd.MultiIndex.from_product([all_dates, sku_list], names=["date", "sku"])
    full_df = pd.DataFrame(index=full_index).reset_index()
    merged = full_df.merge(sales_daily, on=["date", "sku"], how="left")
    merged["qty_sold"] = merged["qty_sold"].fillna(0.0)
    return merged.sort_values(["date", "sku"]).reset_index(drop=True)

def initial_forecast_from_history(demand_series):
    """Use the average of the first few observations as the starting forecast."""
    # A short warmup average gives the agent a stable first forecast.
    warmup_window = min(7, len(demand_series))
    if warmup_window == 0:
        return 0.0
    return float(max(1.0, demand_series.iloc[:warmup_window].mean()))

def update_forecast(previous_forecast, actual_demand, method="ewma", alpha=0.35):
    """Update the next-day forecast using EWMA or naive logic."""
    # Naive forecasting uses yesterday as tomorrow; EWMA smooths recent history.
    if method == "naive":
        return float(actual_demand)
    return float(alpha * actual_demand + (1 - alpha) * previous_forecast)

def z_value_from_service_level(service_level):
    """Convert a target service level into a standard normal z-score."""
    # Clipping avoids invalid extreme values that can break the z-score conversion.
    clipped = min(max(float(service_level), 0.50), 0.999)
    return float(norm.ppf(clipped))

def compute_safety_stock(error_history, service_level, lead_time_days, review_period_days):
    """Compute safety stock using forecast error variability over the risk period."""
    risk_period = max(1, int(lead_time_days) + int(review_period_days))

    # Need a minimum amount of history before using a variability-based buffer.
    if len(error_history) < 5:
        return 0.0

    # Estimate forecast uncertainty from past forecast errors.
    error_std = float(np.std(error_history, ddof=1)) if len(error_history) > 1 else 0.0
    z_value = z_value_from_service_level(service_level)

    # Safety stock increases with both uncertainty and protection window.
    safety_stock = z_value * error_std * math.sqrt(risk_period)
    return float(max(0.0, round(safety_stock, 2)))

print("Forecasting helpers ready.")

Forecasting helpers ready.


---

## 4. Reorder logic, agent role, and policy guardrails

Each review day, the agent:
1. Estimates demand during the **risk period** = lead time + review period
2. Calculates **safety stock**
3. Projects inventory position
4. Decides whether to **ORDER** or **WAIT**

### Guardrails
- Respects `min_order_qty`
- Accounts for `lead_time_days`
- Uses `service_level` in safety stock
- Caps order size to reduce unrealistic over-ordering
- Explains the holding-cost vs stockout-cost trade-off in the log


In [ ]:
# ================================================================
# 4. Reorder decision helpers
# ================================================================
# What this section does:
# - Computes projected inventory and target inventory
# - Applies practical guardrails
# - Returns a human-readable rationale
#
# Ethics / trust angle:
# The point is not just to produce an order quantity. The point is to
# make a decision that can be justified to a planner, manager, or
# instructor in plain language.

def initialize_sku_state(demand_series, opening_stock):
    """Create the mutable simulation state for one SKU."""
    # Store only the variables that change through time for this SKU.
    return {
        "on_hand": float(opening_stock),
        "forecast": initial_forecast_from_history(demand_series),
        "error_history": [],
        "pipeline": {},   # arrival_date -> quantity
    }

def decide_order_for_sku(
    state,
    sku_params,
    current_date,
    review_period_days=1,
    overorder_cap_multiplier=2.5,
):
    """Return order quantity and a brief rationale for a single SKU on the current day."""
    # Read policy inputs for the SKU so one decision can explain all drivers.
    lead_time_days = int(sku_params["lead_time_days"])
    min_order_qty = float(sku_params["min_order_qty"])
    service_level = float(sku_params["service_level"])
    holding_cost = float(sku_params["holding_cost_per_day"])
    stockout_cost = float(sku_params["stockout_cost"])

    risk_period_days = max(1, lead_time_days + review_period_days)
    daily_forecast = float(state["forecast"])

    # The target position covers expected demand across lead time plus review delay.
    demand_over_risk_period = daily_forecast * risk_period_days
    safety_stock = compute_safety_stock(
        error_history=state["error_history"],
        service_level=service_level,
        lead_time_days=lead_time_days,
        review_period_days=review_period_days,
    )
    target_inventory = demand_over_risk_period + safety_stock

    # Inventory position combines what is available now with what is already on order.
    on_hand = float(state["on_hand"])
    in_transit = float(sum(state["pipeline"].values()))
    inventory_position = on_hand + in_transit

    raw_order_qty = max(0.0, target_inventory - inventory_position)

    # Guardrail against unrealistic over-ordering:
    # cap the order at a multiple of expected risk-period demand.
    order_cap = max(min_order_qty, overorder_cap_multiplier * demand_over_risk_period)
    capped_order_qty = min(raw_order_qty, order_cap)

    review_due_today = True  # with daily simulation, review cadence is controlled outside

    if not review_due_today:
        return {
            "action": "WAIT",
            "order_qty": 0.0,
            "reason": "Wait: today is not a review day.",
            "daily_forecast": daily_forecast,
            "risk_period_days": risk_period_days,
            "projected_demand_risk_period": demand_over_risk_period,
            "safety_stock": safety_stock,
            "target_inventory": target_inventory,
            "inventory_position": inventory_position,
            "order_cap": order_cap,
        }

    # If current and incoming stock already cover the target, the agent waits.
    if raw_order_qty <= 0:
        reason = (
            f"Wait: inventory position {inventory_position:.1f} already covers the target {target_inventory:.1f}, "
            f"so ordering now would mainly add holding cost."
        )
        return {
            "action": "WAIT",
            "order_qty": 0.0,
            "reason": reason,
            "daily_forecast": daily_forecast,
            "risk_period_days": risk_period_days,
            "projected_demand_risk_period": demand_over_risk_period,
            "safety_stock": safety_stock,
            "target_inventory": target_inventory,
            "inventory_position": inventory_position,
            "order_cap": order_cap,
        }

    # If the calculated need is too small to meet supplier minimums, the agent waits.
    if capped_order_qty < min_order_qty:
        reason = (
            f"Wait: the need is only {raw_order_qty:.1f} units, which is below the minimum order quantity of {min_order_qty:.1f}."
        )
        return {
            "action": "WAIT",
            "order_qty": 0.0,
            "reason": reason,
            "daily_forecast": daily_forecast,
            "risk_period_days": risk_period_days,
            "projected_demand_risk_period": demand_over_risk_period,
            "safety_stock": safety_stock,
            "target_inventory": target_inventory,
            "inventory_position": inventory_position,
            "order_cap": order_cap,
        }

    # Round the order to a practical whole-unit quantity while respecting MOQ.
    final_order_qty = float(max(min_order_qty, round(capped_order_qty)))

    if raw_order_qty > order_cap:
        reason = (
            f"Order {final_order_qty:.0f}: stock is below target, but the order is capped at {order_cap:.1f} to avoid over-ordering."
        )
    else:
        reason = (
            f"Order {final_order_qty:.0f}: inventory position {inventory_position:.1f} is below the target {target_inventory:.1f} for the risk period."
        )

    return {
        "action": "ORDER",
        "order_qty": final_order_qty,
        "reason": reason,
        "daily_forecast": daily_forecast,
        "risk_period_days": risk_period_days,
        "projected_demand_risk_period": demand_over_risk_period,
        "safety_stock": safety_stock,
        "target_inventory": target_inventory,
        "inventory_position": inventory_position,
        "order_cap": order_cap,
    }

print("Reorder decision helpers ready.")

Reorder decision helpers ready.


---

## 5. Day-by-day simulation engine

This section simulates the full horizon:
- Receives POs after lead time
- Serves demand
- Records stockouts
- Updates forecast
- Lets the agent decide whether to order or wait
- Logs every action in plain text


In [ ]:
# ================================================================
# 5. Simulation engine
# ================================================================
# What this section does:
# - Simulates inventory day by day for each SKU
# - Receives purchase orders on the correct arrival date
# - Fulfills demand, records stockouts, and updates forecasts daily
# - Lets the agent decide whether to order or wait
# - Writes a readable text log that explains each decision
#
# Description:
# The simulation turns the policy into a testable operating process.
# Instead of a static formula, the agent is evaluated as if it were
# making real daily replenishment decisions over time.

def simulate_inventory_agent(
    sales_daily,
    inventory_df,
    params_df,
    forecast_method="ewma",
    alpha=0.35,
    review_period_days=1,
    order_cost_per_po=0.0,
    overorder_cap_multiplier=2.5,
):
    """Run the replenishment agent across the full horizon and return logs plus summary tables."""

    # Start from a copy so the simulation can reshape demand without touching the original table.
    sales_grid = sales_daily.copy()
    sales_grid["date"] = pd.to_datetime(sales_grid["date"])

    # Join inventory and parameter inputs so every SKU carries both stock and policy settings.
    sku_meta = inventory_df.merge(params_df, on="sku", how="inner")
    sku_list = sorted(sku_meta["sku"].unique())

    # Ensure every SKU has one row for every day in the horizon.
    full_dates = pd.DataFrame({"date": sorted(sales_grid["date"].unique())})
    full_index = full_dates.assign(key=1).merge(
        pd.DataFrame({"sku": sku_list, "key": 1}), on="key", how="outer"
    ).drop(columns="key")

    sales_grid = (
        full_index.merge(sales_grid, on=["date", "sku"], how="left")
        .fillna({"qty_sold": 0.0})
        .sort_values(["date", "sku"])
        .reset_index(drop=True)
    )

    # Mutable state for each SKU.
    # Create one independent state container per SKU.
    state = {}
    for sku in sku_list:
        demand_series = sales_grid.loc[sales_grid["sku"] == sku, "qty_sold"].astype(float).reset_index(drop=True)
        opening_stock = float(sku_meta.loc[sku_meta["sku"] == sku, "opening_stock"].iloc[0])
        state[sku] = initialize_sku_state(demand_series=demand_series, opening_stock=opening_stock)

    daily_records = []
    text_log_lines = []
    unique_dates = sorted(sales_grid["date"].unique())
    total_days = len(unique_dates)

    # Walk forward one simulated day at a time so decisions happen in sequence.
    for day_index, current_date in enumerate(unique_dates):
        current_date = pd.Timestamp(current_date)
        day_number = day_index + 1
        is_review_day = (day_index % review_period_days == 0)

        text_log_lines.append("." * 110)
        text_log_lines.append(
            f"DAY {day_number} OF {total_days} | {current_date.date()} | review check: {'yes' if is_review_day else 'no'}"
        )
        text_log_lines.append("." * 110)

        for sku in sku_list:
            sku_params = sku_meta.loc[sku_meta["sku"] == sku].iloc[0]
            sku_state = state[sku]

            # Snapshot beginning inventory for transparent logging and cost tracking.
            on_hand_start = float(sku_state["on_hand"])

            # Receive purchase orders first.
            arrivals_today = float(sku_state["pipeline"].pop(current_date, 0.0))
            sku_state["on_hand"] += arrivals_today
            on_hand_after_arrivals = float(sku_state["on_hand"])

            # Pull the realized demand for this specific date and SKU.
            demand_today = float(
                sales_grid.loc[
                    (sales_grid["date"] == current_date) & (sales_grid["sku"] == sku), "qty_sold"
                ].iloc[0]
            )

            # Demand is served immediately from available stock; the remainder becomes a stockout.
            fulfilled_units = min(float(sku_state["on_hand"]), demand_today)
            stockout_units = max(0.0, demand_today - fulfilled_units)
            sku_state["on_hand"] -= fulfilled_units
            on_hand_end = float(sku_state["on_hand"])

            # Compare actual demand with the prior forecast to update error history.
            forecast_used_today = float(sku_state["forecast"])
            forecast_error = float(demand_today - forecast_used_today)
            sku_state["error_history"].append(forecast_error)

            if is_review_day:
                order_action = decide_order_for_sku(
                    state=sku_state,
                    sku_params=sku_params,
                    current_date=current_date,
                    review_period_days=review_period_days,
                    overorder_cap_multiplier=overorder_cap_multiplier,
                )
            else:
                order_action = {
                    "action": "WAIT",
                    "order_qty": 0.0,
                    "reason": "Wait because this day is outside the review cadence, so the agent avoids unnecessary order checks.",
                    "daily_forecast": forecast_used_today,
                    "risk_period_days": np.nan,
                    "projected_demand_risk_period": np.nan,
                    "safety_stock": np.nan,
                    "target_inventory": np.nan,
                    "inventory_position": np.nan,
                    "order_cap": np.nan,
                }

            # If an order is placed, register its future arrival in the pipeline.
            if order_action["action"] == "ORDER":
                arrival_date = current_date + pd.Timedelta(days=int(sku_params["lead_time_days"]))
                sku_state["pipeline"][arrival_date] = sku_state["pipeline"].get(arrival_date, 0.0) + float(order_action["order_qty"])
            else:
                arrival_date = pd.NaT

            # Update the forecast after seeing today's demand so tomorrow starts with fresh information.
            next_forecast = update_forecast(
                previous_forecast=forecast_used_today,
                actual_demand=demand_today,
                method=forecast_method,
                alpha=alpha,
            )
            sku_state["forecast"] = float(next_forecast)

            in_transit_after_decision = float(sum(sku_state["pipeline"].values()))
            holding_cost_today = float(sku_state["on_hand"] * float(sku_params["holding_cost_per_day"]))
            stockout_cost_today = float(stockout_units * float(sku_params["stockout_cost"]))
            order_cost_today = float(order_cost_per_po if order_action["action"] == "ORDER" else 0.0)
            total_cost_today = holding_cost_today + stockout_cost_today + order_cost_today

            # Store a machine-readable record for later summaries and CSV export.
            daily_records.append({
                "date": current_date.date(),
                "sku": sku,
                "review_day": is_review_day,
                "on_hand_start": round(on_hand_start, 2),
                "arrivals_today": round(arrivals_today, 2),
                "on_hand_after_arrivals": round(on_hand_after_arrivals, 2),
                "demand_today": round(demand_today, 2),
                "fulfilled_units": round(fulfilled_units, 2),
                "stockout_units": round(stockout_units, 2),
                "on_hand_end": round(on_hand_end, 2),
                "forecast_used_today": round(forecast_used_today, 2),
                "forecast_for_next_day": round(float(next_forecast), 2),
                "forecast_error": round(forecast_error, 2),
                "projected_demand_risk_period": round(float(order_action["projected_demand_risk_period"]), 2) if pd.notna(order_action["projected_demand_risk_period"]) else np.nan,
                "safety_stock": round(float(order_action["safety_stock"]), 2) if pd.notna(order_action["safety_stock"]) else np.nan,
                "target_inventory": round(float(order_action["target_inventory"]), 2) if pd.notna(order_action["target_inventory"]) else np.nan,
                "inventory_position": round(float(order_action["inventory_position"]), 2) if pd.notna(order_action["inventory_position"]) else np.nan,
                "action": order_action["action"],
                "order_qty": round(float(order_action["order_qty"]), 2),
                "arrival_date": arrival_date.date() if pd.notna(arrival_date) else "",
                "in_transit_after_decision": round(in_transit_after_decision, 2),
                "holding_cost_today": round(holding_cost_today, 2),
                "stockout_cost_today": round(stockout_cost_today, 2),
                "order_cost_today": round(order_cost_today, 2),
                "total_cost_today": round(total_cost_today, 2),
                "decision_reason": order_action["reason"],
            })

            projected_risk = order_action["projected_demand_risk_period"]
            risk_text = f"{float(projected_risk):.1f}" if pd.notna(projected_risk) else "n/a"
            risk_days_text = f"{int(order_action['risk_period_days'])}" if pd.notna(order_action["risk_period_days"]) else "n/a"
            service_level_today = float(sku_params["service_level"])
            z_value_today = float(norm.ppf(service_level_today)) if 0 < service_level_today < 1 else np.nan

            # Also store a plain-English trace so a reader can follow the rationale day by day.
            text_log_lines.append(f"SKU {sku}")
            text_log_lines.append(
                f"  Demand flow      : requested {demand_today:.1f} | fulfilled {fulfilled_units:.1f} | stockout {stockout_units:.1f} | ending on-hand {on_hand_end:.1f}"
            )
            text_log_lines.append(
                f"  Forecast view    : today {forecast_used_today:.1f} units | risk window {risk_days_text} day(s) | projected risk demand {risk_text}"
            )

            if pd.notna(order_action["safety_stock"]):
                text_log_lines.append(
                    f"  Safety buffer    : service level {service_level_today:.2%} | z {z_value_today:.2f} | safety stock {float(order_action['safety_stock']):.1f} | target position {float(order_action['target_inventory']):.1f}"
                )
                text_log_lines.append(
                    f"  Inventory status : position {float(order_action['inventory_position']):.1f} | in transit after decision {in_transit_after_decision:.1f}"
                )
            else:
                text_log_lines.append("  Safety buffer    : not evaluated today because this was not a review day.")
                text_log_lines.append(
                    f"  Inventory status : position not reviewed today | in transit after decision {in_transit_after_decision:.1f}"
                )

            if order_action["action"] == "ORDER":
                text_log_lines.append(
                    f"  Agent decision   : ORDER {float(order_action['order_qty']):.1f} units | expected arrival {arrival_date.date()}"
                )
            else:
                text_log_lines.append("  Agent decision   : WAIT | no purchase order released today")

            text_log_lines.append(f"  Why              : {order_action['reason']}")
            text_log_lines.append(
                f"  Cost snapshot    : holding ${holding_cost_today:.2f} | stockout ${stockout_cost_today:.2f} | order ${order_cost_today:.2f}"
            )
            text_log_lines.append(
                f"  Forecast update  : tomorrow's starting forecast becomes {float(next_forecast):.1f} units"
            )
            text_log_lines.append("")

    # Convert the raw event list into tables for reporting and evaluation.
    daily_log_df = pd.DataFrame(daily_records)

    # Summarize performance per SKU so the final report can show operational patterns.
    sku_summary = (
        daily_log_df.groupby("sku", as_index=False)
        .agg(
            total_demand=("demand_today", "sum"),
            fulfilled_units=("fulfilled_units", "sum"),
            stockout_units=("stockout_units", "sum"),
            holding_cost=("holding_cost_today", "sum"),
            stockout_cost=("stockout_cost_today", "sum"),
            order_cost=("order_cost_today", "sum"),
            total_orders=("action", lambda x: int((x == "ORDER").sum())),
            wait_days=("action", lambda x: int((x == "WAIT").sum())),
            total_order_qty=("order_qty", "sum"),
            ending_inventory=("on_hand_end", "last"),
        )
    )

    sku_summary["fill_rate"] = np.where(
        sku_summary["total_demand"] > 0,
        sku_summary["fulfilled_units"] / sku_summary["total_demand"],
        1.0,
    )
    sku_summary["total_cost"] = sku_summary["holding_cost"] + sku_summary["stockout_cost"] + sku_summary["order_cost"]

    final_inventory_on_hand = float(sku_summary["ending_inventory"].sum()) if not sku_summary.empty else 0.0
    total_stockout_days = int(daily_log_df.groupby("date")["stockout_units"].sum().gt(0).sum()) if not daily_log_df.empty else 0
    final_backorders = 0.0

    # Build one overall summary row for the full system horizon.
    total_summary = pd.DataFrame([{
        "total_demand": round(float(daily_log_df["demand_today"].sum()), 2),
        "fulfilled_units": round(float(daily_log_df["fulfilled_units"].sum()), 2),
        "stockout_units": round(float(daily_log_df["stockout_units"].sum()), 2),
        "fill_rate": round(float(daily_log_df["fulfilled_units"].sum() / daily_log_df["demand_today"].sum()), 4) if daily_log_df["demand_today"].sum() > 0 else 1.0,
        "total_stockout_days": total_stockout_days,
        "final_inventory_on_hand": round(final_inventory_on_hand, 2),
        "final_backorders": round(final_backorders, 2),
        "holding_cost": round(float(daily_log_df["holding_cost_today"].sum()), 2),
        "stockout_cost": round(float(daily_log_df["stockout_cost_today"].sum()), 2),
        "order_cost": round(float(daily_log_df["order_cost_today"].sum()), 2),
        "total_cost": round(float(daily_log_df["total_cost_today"].sum()), 2),
        "orders_placed": int((daily_log_df["action"] == "ORDER").sum()),
        "total_order_qty": round(float(daily_log_df["order_qty"].sum()), 2),
    }])

    # Round key summary fields for cleaner final printing.
    for col in ["holding_cost", "stockout_cost", "order_cost", "total_cost", "total_order_qty", "ending_inventory"]:
        if col in sku_summary.columns:
            sku_summary[col] = sku_summary[col].astype(float).round(2)
    if "fill_rate" in sku_summary.columns:
        sku_summary["fill_rate"] = sku_summary["fill_rate"].astype(float).round(4)

    return daily_log_df, sku_summary, total_summary, text_log_lines

print("Simulation engine ready.")

Simulation engine ready.


---

## Part 6. Run the inventory replenishment agent


This step runs the agent and prints a cleaner text report for the full simulation horizon. It now:
- Summarizes the system-wide demand, fulfillment, stockout, and cost results
- States the agent role, decision inputs, outputs, guardrails, and ethics/trust links
- Adds an overall 90-day order-versus-wait summary by SKU
- Prints a day-by-day rationale in a more structured plain-text format

The goal is to make the notebook easier to read while still showing exactly why the agent ordered or waited on each day.


In [ ]:
# ================================================================
# 6. Run the agent and print organized outputs
# ================================================================
# What this section does:
# - Runs the inventory replenishment agent across the full horizon
# - Runs a simple baseline policy for comparison
# - Prints a structured business summary first, followed by the decision log
# - Explains the agent role, guardrails, and metrics in plain language
# - Shows a 90-day order-versus-wait summary by SKU
#
# Description:
# The output should be readable by a business stakeholder, not just by a
# programmer. Clear summaries and concise rationales make the agent easier
# to audit, easier to trust, and easier to explain in class.

# ----------------------------------------------------------------
# Baseline policy helper
# ----------------------------------------------------------------
# What this helper does:
# - Creates a simple comparison policy with fewer decision features
# - Uses a fixed average daily demand per SKU instead of a dynamic forecast
# - Uses a fixed reorder point and order-up-to target with no safety stock
#
# Why this baseline matters:
# A baseline makes the optimized agent easier to evaluate. Instead of only
# showing final numbers, it shows whether the more thoughtful forecasting
# and safety-stock logic actually improved service or cost outcomes, using
# modifiers to help simulate projected insight vs. realism.
def simulate_baseline_policy(
    sales_daily,
    inventory_df,
    params_df,
    review_period_days=1,
    order_cost_per_po=0.0,
):
    """Run a simple fixed-rule replenishment baseline for comparison."""

    # Copy and standardize the demand table so the baseline uses the same input horizon.
    sales_grid = sales_daily.copy()
    sales_grid["date"] = pd.to_datetime(sales_grid["date"])

    # Merge inventory and parameter tables so each SKU carries all policy settings.
    sku_meta = inventory_df.merge(params_df, on="sku", how="inner")
    sku_list = sorted(sku_meta["sku"].unique())

    # Expand the demand table so every SKU has one row on every date.
    full_dates = pd.DataFrame({"date": sorted(sales_grid["date"].unique())})
    full_index = full_dates.assign(key=1).merge(
        pd.DataFrame({"sku": sku_list, "key": 1}), on="key", how="outer"
    ).drop(columns="key")

    sales_grid = (
        full_index.merge(sales_grid, on=["date", "sku"], how="left")
        .fillna({"qty_sold": 0.0})
        .sort_values(["date", "sku"])
        .reset_index(drop=True)
    )

    # The baseline keeps a much simpler state: stock on hand and open purchase orders.
    state = {}
    for sku in sku_list:
        demand_series = sales_grid.loc[sales_grid["sku"] == sku, "qty_sold"].astype(float).reset_index(drop=True)
        opening_stock = float(sku_meta.loc[sku_meta["sku"] == sku, "opening_stock"].iloc[0])

        # Fixed average demand becomes the baseline's only demand estimate.
        static_daily_demand = float(demand_series.mean()) if len(demand_series) > 0 else 0.0
        state[sku] = {
            "on_hand": opening_stock,
            "pipeline": {},
            "static_daily_demand": static_daily_demand,
        }

    daily_records = []
    unique_dates = sorted(sales_grid["date"].unique())

    # Simulate day by day so the baseline faces the same realized demand stream as the agent.
    for day_index, current_date in enumerate(unique_dates):
        current_date = pd.Timestamp(current_date)
        is_review_day = (day_index % review_period_days == 0)

        for sku in sku_list:
            sku_params = sku_meta.loc[sku_meta["sku"] == sku].iloc[0]
            sku_state = state[sku]

            on_hand_start = float(sku_state["on_hand"])
            arrivals_today = float(sku_state["pipeline"].pop(current_date, 0.0))
            sku_state["on_hand"] += arrivals_today
            on_hand_after_arrivals = float(sku_state["on_hand"])

            demand_today = float(
                sales_grid.loc[
                    (sales_grid["date"] == current_date) & (sales_grid["sku"] == sku), "qty_sold"
                ].iloc[0]
            )

            fulfilled_units = min(float(sku_state["on_hand"]), demand_today)
            stockout_units = max(0.0, demand_today - fulfilled_units)
            sku_state["on_hand"] -= fulfilled_units
            on_hand_end = float(sku_state["on_hand"])

            lead_time_days = int(sku_params["lead_time_days"])
            min_order_qty = float(sku_params["min_order_qty"])
            static_daily_demand = float(sku_state["static_daily_demand"])

            # The baseline ignores forecast error and service-level safety stock.
            reorder_point = static_daily_demand * max(1, lead_time_days)
            target_inventory = static_daily_demand * max(1, lead_time_days + review_period_days)
            inventory_position = float(sku_state["on_hand"] + sum(sku_state["pipeline"].values()))

            order_qty = 0.0
            action = "WAIT"
            if is_review_day and inventory_position < reorder_point:
                order_need = max(0.0, target_inventory - inventory_position)
                if order_need > 0:
                    order_qty = float(max(min_order_qty, round(order_need)))
                    action = "ORDER"
                    arrival_date = current_date + pd.Timedelta(days=lead_time_days)
                    sku_state["pipeline"][arrival_date] = sku_state["pipeline"].get(arrival_date, 0.0) + order_qty
                else:
                    arrival_date = pd.NaT
            else:
                arrival_date = pd.NaT

            in_transit_after_decision = float(sum(sku_state["pipeline"].values()))
            holding_cost_today = float(sku_state["on_hand"] * float(sku_params["holding_cost_per_day"]))
            stockout_cost_today = float(stockout_units * float(sku_params["stockout_cost"]))
            order_cost_today = float(order_cost_per_po if action == "ORDER" else 0.0)
            total_cost_today = holding_cost_today + stockout_cost_today + order_cost_today

            daily_records.append({
                "date": current_date.date(),
                "sku": sku,
                "review_day": is_review_day,
                "on_hand_start": round(on_hand_start, 2),
                "arrivals_today": round(arrivals_today, 2),
                "on_hand_after_arrivals": round(on_hand_after_arrivals, 2),
                "demand_today": round(demand_today, 2),
                "fulfilled_units": round(fulfilled_units, 2),
                "stockout_units": round(stockout_units, 2),
                "on_hand_end": round(on_hand_end, 2),
                "forecast_used_today": round(static_daily_demand, 2),
                "forecast_for_next_day": round(static_daily_demand, 2),
                "forecast_error": round(demand_today - static_daily_demand, 2),
                "projected_demand_risk_period": round(float(target_inventory), 2),
                "safety_stock": 0.0,
                "target_inventory": round(float(target_inventory), 2),
                "inventory_position": round(float(inventory_position), 2),
                "action": action,
                "order_qty": round(float(order_qty), 2),
                "arrival_date": arrival_date.date() if pd.notna(arrival_date) else "",
                "in_transit_after_decision": round(in_transit_after_decision, 2),
                "holding_cost_today": round(holding_cost_today, 2),
                "stockout_cost_today": round(stockout_cost_today, 2),
                "order_cost_today": round(order_cost_today, 2),
                "total_cost_today": round(total_cost_today, 2),
                "decision_reason": "Baseline fixed-rule policy: order only when inventory position drops below a simple lead-time reorder point." if action == "ORDER" else "Baseline fixed-rule policy: wait because inventory position still covers the simple reorder point.",
            })

    baseline_daily_log = pd.DataFrame(daily_records)

    baseline_sku_summary = (
        baseline_daily_log.groupby("sku", as_index=False)
        .agg(
            total_demand=("demand_today", "sum"),
            fulfilled_units=("fulfilled_units", "sum"),
            stockout_units=("stockout_units", "sum"),
            holding_cost=("holding_cost_today", "sum"),
            stockout_cost=("stockout_cost_today", "sum"),
            order_cost=("order_cost_today", "sum"),
            total_orders=("action", lambda x: int((x == "ORDER").sum())),
            wait_days=("action", lambda x: int((x == "WAIT").sum())),
            total_order_qty=("order_qty", "sum"),
            ending_inventory=("on_hand_end", "last"),
        )
    )

    baseline_sku_summary["fill_rate"] = np.where(
        baseline_sku_summary["total_demand"] > 0,
        baseline_sku_summary["fulfilled_units"] / baseline_sku_summary["total_demand"],
        1.0,
    )
    baseline_sku_summary["total_cost"] = (
        baseline_sku_summary["holding_cost"]
        + baseline_sku_summary["stockout_cost"]
        + baseline_sku_summary["order_cost"]
    )

    final_inventory_on_hand = float(baseline_sku_summary["ending_inventory"].sum()) if not baseline_sku_summary.empty else 0.0
    total_stockout_days = int(baseline_daily_log.groupby("date")["stockout_units"].sum().gt(0).sum()) if not baseline_daily_log.empty else 0

    baseline_total_summary = pd.DataFrame([{
        "total_demand": round(float(baseline_daily_log["demand_today"].sum()), 2),
        "fulfilled_units": round(float(baseline_daily_log["fulfilled_units"].sum()), 2),
        "stockout_units": round(float(baseline_daily_log["stockout_units"].sum()), 2),
        "fill_rate": round(float(baseline_daily_log["fulfilled_units"].sum() / baseline_daily_log["demand_today"].sum()), 4) if baseline_daily_log["demand_today"].sum() > 0 else 1.0,
        "total_stockout_days": total_stockout_days,
        "final_inventory_on_hand": round(final_inventory_on_hand, 2),
        "final_backorders": 0.0,
        "holding_cost": round(float(baseline_daily_log["holding_cost_today"].sum()), 2),
        "stockout_cost": round(float(baseline_daily_log["stockout_cost_today"].sum()), 2),
        "order_cost": round(float(baseline_daily_log["order_cost_today"].sum()), 2),
        "total_cost": round(float(baseline_daily_log["total_cost_today"].sum()), 2),
        "orders_placed": int((baseline_daily_log["action"] == "ORDER").sum()),
        "total_order_qty": round(float(baseline_daily_log["order_qty"].sum()), 2),
    }])

    for col in ["holding_cost", "stockout_cost", "order_cost", "total_cost", "total_order_qty", "ending_inventory"]:
        if col in baseline_sku_summary.columns:
            baseline_sku_summary[col] = baseline_sku_summary[col].astype(float).round(2)
    if "fill_rate" in baseline_sku_summary.columns:
        baseline_sku_summary["fill_rate"] = baseline_sku_summary["fill_rate"].astype(float).round(4)

    return baseline_daily_log, baseline_sku_summary, baseline_total_summary

# Run the optimized agent using the selected policy parameters.
agent_daily_log, agent_sku_summary, agent_total_summary, agent_text_log = simulate_inventory_agent(
    sales_daily=sales_daily,
    inventory_df=inventory_df,
    params_df=params_df,
    forecast_method=FORECAST_METHOD,
    alpha=ALPHA,
    review_period_days=REVIEW_PERIOD_DAYS,
    order_cost_per_po=ORDER_COST_PER_PO,
    overorder_cap_multiplier=OVERORDER_CAP_MULTIPLIER,
)

# Run the simple baseline so the final report can compare both approaches.
baseline_daily_log, baseline_sku_summary, baseline_total_summary = simulate_baseline_policy(
    sales_daily=sales_daily,
    inventory_df=inventory_df,
    params_df=params_df,
    review_period_days=REVIEW_PERIOD_DAYS,
    order_cost_per_po=ORDER_COST_PER_PO,
)

# Small helper functions keep report formatting consistent and readable.
def print_kv(label, value, width=28):
    print(f"{label:<{width}} {value}")

def fmt_num(value, decimals=2):
    return f"{float(value):,.{decimals}f}"

def fmt_money(value):
    return f"${float(value):,.2f}"

# Pull frequently used report values into local variables for simpler printing code.
summary_row = agent_total_summary.iloc[0]
baseline_row = baseline_total_summary.iloc[0]
total_days_simulated = int(agent_daily_log["date"].nunique())
total_skus = int(agent_daily_log["sku"].nunique())
line = "." * 98

# This summary shows how often each SKU triggered an order versus a wait decision.
action_summary = (
    agent_daily_log.groupby("sku", as_index=False)
    .agg(
        order_days=("action", lambda x: int((x == "ORDER").sum())),
        wait_days=("action", lambda x: int((x == "WAIT").sum())),
        total_order_qty=("order_qty", "sum"),
        total_stockout_units=("stockout_units", "sum"),
        ending_inventory=("on_hand_end", "last"),
    )
)
action_summary["total_order_qty"] = action_summary["total_order_qty"].round(2)
action_summary["total_stockout_units"] = action_summary["total_stockout_units"].round(2)
action_summary["ending_inventory"] = action_summary["ending_inventory"].round(2)

# Build a compact comparison table so the optimized policy can be contrasted with the baseline.
comparison_rows = [
    ("Fill Rate (Orders Met)", f"{float(summary_row['fill_rate']) * 100:,.2f}%", f"{float(baseline_row['fill_rate']) * 100:,.2f}%"),
    ("Total Stockout Units", f"{fmt_num(summary_row['stockout_units'])} units", f"{fmt_num(baseline_row['stockout_units'])} units"),
    ("Total Stockout Days", f"{int(summary_row['total_stockout_days'])}", f"{int(baseline_row['total_stockout_days'])}"),
    ("Final Inventory On Hand", f"{fmt_num(summary_row['final_inventory_on_hand'])} units", f"{fmt_num(baseline_row['final_inventory_on_hand'])} units"),
    ("Total Holding Cost", fmt_money(summary_row['holding_cost']), fmt_money(baseline_row['holding_cost'])),
    ("Total Stockout Cost", fmt_money(summary_row['stockout_cost']), fmt_money(baseline_row['stockout_cost'])),
    ("Total Order Cost (Fixed)", fmt_money(summary_row['order_cost']), fmt_money(baseline_row['order_cost'])),
    ("TOTAL SYSTEM COST", fmt_money(summary_row['total_cost']), fmt_money(baseline_row['total_cost'])),
]

# Compute directional improvements so the reader can see whether the optimized agent helped.
fill_rate_delta_pct = (float(summary_row['fill_rate']) - float(baseline_row['fill_rate'])) * 100
stockout_unit_delta = float(baseline_row['stockout_units']) - float(summary_row['stockout_units'])
total_cost_delta = float(baseline_row['total_cost']) - float(summary_row['total_cost'])

print("" + line)
print("INVENTORY REPLENISHMENT AGENT REPORT")
print(line)
print_kv("Simulation horizon:", f"{total_days_simulated} days")
print_kv("SKUs managed:", f"{total_skus}")
print_kv("Forecast method:", FORECAST_METHOD.upper())
print_kv("Review period:", f"{REVIEW_PERIOD_DAYS} day(s)")
print_kv("Order cost per PO:", fmt_money(ORDER_COST_PER_PO))

print("" + line)
print("AGENT ROLE")
print(line)
print("The agent minimizes stockouts and total inventory cost. Each review decision uses demand forecasts,")
print("current stock, incoming orders, lead time, service level, and cost parameters to choose one of two")
print("outputs: place a replenishment order or wait and avoid unnecessary holding cost.")
print("Decision inputs : forecast, inventory on hand, pipeline stock, lead time, service level, min order quantity")
print("Decision outputs: ORDER a quantity that respects policy guardrails, or WAIT when inventory already covers risk")

print("" + line)
print("GUARDRAILS, BUSINESS RULES, ETHICS, AND TRUST")
print(line)
print("- Minimum order quantity is always respected so the recommendation stays operationally realistic.")
print("- Lead time is built into the risk window so the agent does not assume instant replenishment.")
print("- Service level shapes safety stock so the policy reflects reliability goals, not guesswork.")
print("- Over-ordering is capped to avoid tying up capital in unrealistic inventory builds.")
print("- Holding cost and stockout cost are both considered so the agent balances efficiency with service.")
print("Ethics and trust link: customers are harmed by avoidable stockouts, while the business is harmed by")
print("excess inventory and wasted cash. Transparent rules and daily explanations make the policy accountable.")

print("" + line)
print("OVERALL RESULTS AFTER THE FULL HORIZON")
print(line)
print_kv("Total Demand:", f"{fmt_num(summary_row['total_demand'])} units")
print_kv("Total Fulfilled:", f"{fmt_num(summary_row['fulfilled_units'])} units")
print_kv("Fill Rate (Orders Met):", f"{float(summary_row['fill_rate']) * 100:,.2f}%")
print_kv("Total Stockout Units:", f"{fmt_num(summary_row['stockout_units'])} units")
print_kv("Total Stockout Days:", f"{int(summary_row['total_stockout_days'])}")
print_kv("Final Inventory On Hand:", f"{fmt_num(summary_row['final_inventory_on_hand'])} units")
print_kv("Final Backorders:", f"{fmt_num(summary_row['final_backorders'])} units")

print("[ FINANCIAL METRICS ]")
print_kv("Total Holding Cost:", fmt_money(summary_row['holding_cost']))
print_kv("Total Stockout Cost:", fmt_money(summary_row['stockout_cost']))
print_kv("Total Order Cost (Fixed):", fmt_money(summary_row['order_cost']))
print_kv("TOTAL SYSTEM COST:", fmt_money(summary_row['total_cost']))

print("" + line)
print(f"ORDER OR WAIT SUMMARY AFTER {total_days_simulated} DAYS")
print(line)
for _, row in action_summary.iterrows():
    print(
        f"{row['sku']}: order days={int(row['order_days'])}, wait days={int(row['wait_days'])}, "
        f"total ordered={row['total_order_qty']:.2f}, total stockout units={row['total_stockout_units']:.2f}, "
        f"ending inventory={row['ending_inventory']:.2f}"
    )

print("" + line)
print("BASELINE COMPARISON")
print(line)
print("Baseline policy used: a simple fixed reorder rule with static average demand, lead-time reorder point,")
print("and no dynamic forecast updates or safety stock. This keeps the comparison intentionally simple.")
for metric_label, agent_value, baseline_value in comparison_rows:
    print(f"{metric_label:<28} Agent={agent_value:<18} | Baseline={baseline_value}")

print("Comparison highlights:")
print(f"- Fill-rate change vs baseline : {fill_rate_delta_pct:+.2f} percentage points")
print(f"- Stockout-unit change         : {stockout_unit_delta:+.2f} units")
print(f"- Total-cost change            : {fmt_money(total_cost_delta)} (positive means agent saved money)")

print("" + line)
print("METRICS, ETHICS, AND TRUST")
print(line)
print("Stockout units show how much demand the system could not serve; lower is better for customer trust.")
print("Fill rate shows the share of demand fulfilled; higher is better for service reliability and credibility.")
print("Total system cost combines holding, stockout, and order cost; lower is better for responsible operations.")
print("Comparing the optimized policy against a simpler baseline helps show whether the added complexity")
print("actually earns its place through better service, lower cost, or a clearer trade-off between both.")
print("The daily log below explains why the agent ordered or waited, which supports transparency and auditability.")

# Print the detailed day-by-day rationale only when full logging is enabled.
if PRINT_FULL_DAILY_LOG and not PRINT_SUMMARY_ONLY:
    print("" + line)
    print("DAILY AGENT DECISION LOG")
    print(line)
    print(f"The log spans {total_days_simulated} total days. Each header is labeled as DAY X OF {total_days_simulated}.")
    for line_text in agent_text_log:
        print(line_text)



..................................................................................................
INVENTORY REPLENISHMENT AGENT REPORT
..................................................................................................
Simulation horizon:          90 days
SKUs managed:                3
Forecast method:             EWMA
Review period:               1 day(s)
Order cost per PO:           $0.00
..................................................................................................
AGENT ROLE
..................................................................................................
The agent minimizes stockouts and total inventory cost. Each review decision uses demand forecasts,
current stock, incoming orders, lead time, service level, and cost parameters to choose one of two
outputs: place a replenishment order or wait and avoid unnecessary holding cost.
Decision inputs : forecast, inventory on hand, pipeline stock, lead time, service level, min order qu

---

## 7. Save outputs (Optional)

This section is optional, which writes the useful outputs to CSV so you can submit, inspect, or visualize them later.


In [ ]:
# ================================================================
# 7. Save outputs
# ================================================================
# What this section does:
# - Saves the detailed daily log
# - Saves SKU-level and total summaries
# - Saves baseline comparison outputs as separate CSV files
#
# Description:
# Saving outputs supports reproducibility and lets you inspect the agent
# decisions outside the notebook if needed.

# Persist the main outputs so they can be reused outside the notebook.
if SAVE_OUTPUTS:
    agent_daily_log.to_csv("agent_daily_log_clean.csv", index=False)
    agent_sku_summary.to_csv("agent_sku_summary_clean.csv", index=False)
    agent_total_summary.to_csv("agent_total_summary_clean.csv", index=False)
    baseline_daily_log.to_csv("baseline_daily_log_clean.csv", index=False)
    baseline_sku_summary.to_csv("baseline_sku_summary_clean.csv", index=False)
    baseline_total_summary.to_csv("baseline_total_summary_clean.csv", index=False)

    print("Saved output files:")
    print("- agent_daily_log_clean.csv")
    print("- agent_sku_summary_clean.csv")
    print("- agent_total_summary_clean.csv")
    print("- baseline_daily_log_clean.csv")
    print("- baseline_sku_summary_clean.csv")
    print("- baseline_total_summary_clean.csv")
else:
    print("SAVE_OUTPUTS is False, so no CSV outputs were written.")



Saved output files:
- agent_daily_log_clean.csv
- agent_sku_summary_clean.csv
- agent_total_summary_clean.csv
